# Week 9: Data Cleaning

**IT 2012 — Unstructured Data**


Last week we explored our extracted-documents dataset and discovered the mess:
missing values, inconsistent strings, whitespace in author names, mixed languages,
and fields with the wrong data type. This week we **clean it** — systematically,
reproducibly, and with tests to prove it worked.

**What you will learn:**

1. Assess data quality across five dimensions
2. Handle missing data: drop, fill, interpolate — and know *when* each is appropriate
3. Transform data with `map`, `apply`, `replace`, and lambda functions
4. Clean strings using the pandas `.str` accessor and regular expressions
5. Detect and remove duplicates (exact and fuzzy)
6. Convert data types correctly (`astype`, `to_datetime`, `to_numeric`)
7. Validate cleaned data with assertions
8. Write simple tests with **pytest**
9. Build a reproducible cleaning pipeline as a reusable function

**Prerequisites:** Week 8 notebook completed; `extracted_documents.json` in the same folder.

---
## 1 — Setup and data loading

We reload the same 60-document JSON from Week 8 and flatten it with
`pd.json_normalize`, exactly as before. This gives us our starting point —
the messy DataFrame we ended Week 8 with.

In [1]:
import numpy as np
import pandas as pd
import json
import re
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 120)

print(f"NumPy  {np.__version__}")
print(f"pandas {pd.__version__}")

NumPy  2.3.5
pandas 2.3.3


In [2]:
# ── Load JSON (same file from Week 8) ──────────────────────────
# In production this would be:
#   docs = list(client.unstructured.documents.find())
# We simulate it with a JSON file.

with open("extracted_documents_week9.json", "r", encoding="utf-8") as f:
    docs = json.load(f)

print(f"Loaded {len(docs)} documents")

Loaded 60 documents


In [3]:
# Flatten nested MongoDB-style documents into a flat table
df_raw = pd.json_normalize(docs)
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
df_raw.head(3)

Shape: (60, 13)
Columns: ['_id', 'title', 'source_type', 'extracted_at', 'file_size_kb', 'language', 'status', 'content.text', 'content.word_count', 'metadata.author', 'metadata.pages', 'metadata.duration_sec', 'metadata.ocr_confidence']


,_id,title,source_type,extracted_at,file_size_kb,language,status,content.text,content.word_count,metadata.author,metadata.pages,metadata.duration_sec,metadata.ocr_confidence
0,3d6d5f37ab13117b9f40eb74,Annual Report 2024 — IBU,pdf,2025-02-11T03:00:00,283.4,en,processed,Univerzitet je osnovao program za razmjenu studenata sa ...,10,"Hodžić, Amina",48.0,NaN,NaN
1,4bd4f2e5596c6d73a1b09913,Campus Map Scan,image,2024-09-27T21:00:00,517.9,bs,extracted,The algorithm achieved 94.2% accuracy on the test set.,9,"Smith, John",NaN,NaN,0.61
2,ab913cf6186525361ed6a2a6,Lecture Recording — Week 3,audio,2024-10-26T07:00:00,682.3,bs,extracted,Sarajevo's Old Town features Ottoman-era architecture da...,11,"Kovačević, Marko",NaN,2586.2,NaN


In [4]:
# IMPORTANT: keep an untouched copy so we can compare before/after
df = df_raw.copy()
print("Working copy created. We never touch df_raw again.")

Working copy created. We never touch df_raw again.


---
## 2 — Why data cleaning matters

> **"Garbage in, garbage out."**

Data scientists spend roughly **60–80 %** of their time cleaning data.
If you skip this step, every downstream result — statistics, models,
dashboards — will be wrong or misleading.

Data cleaning is not optional grunt work. It is analysis. Every cleaning
decision encodes a judgement about the domain.

### 2.1 Five data-quality dimensions

| Dimension | Question | Example in our data |
|-----------|----------|---------------------|
| **Completeness** | Are values present? | `file_size_kb` missing for 3 docs |
| **Accuracy** | Are values correct? | A `word_count` of −5 would be wrong |
| **Consistency** | Same format everywhere? | `"en"` vs `"EN"` vs `"English"` |
| **Timeliness** | Is data current enough? | `extracted_at` dates from the future? |
| **Validity** | Within expected range? | Negative page counts, impossible dates |

We will check all five for our dataset.

---
## 3 — Data quality audit

Before cleaning, **measure the mess**. Run a structured audit so you know
exactly what needs fixing and can verify improvements later.

In [5]:
# 3.1  Overall missing-value report
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(1)

audit = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct,
    "dtype": df.dtypes
})
audit[audit["missing_count"] > 0]

,missing_count,missing_pct,dtype
file_size_kb,3,5.0,float64
language,10,16.7,object
metadata.author,4,6.7,object
metadata.duration_sec,45,75.0,float64
metadata.ocr_confidence,45,75.0,float64
metadata.pages,45,75.0,float64
status,8,13.3,object


In [6]:
# 3.2  Quick quality snapshot — numeric columns
print("=== Numeric summary ===")
df.describe()

=== Numeric summary ===


,file_size_kb,content.word_count,metadata.pages,metadata.duration_sec,metadata.ocr_confidence
count,57.000000,60.000000,15.000000,15.000000,15.000000
mean,652.501754,9.550000,27.933333,1930.893333,0.772667
std,343.292406,1.577651,17.774246,1046.792378,0.123836
min,24.200000,6.000000,5.000000,209.000000,0.610000
25%,332.900000,8.750000,11.000000,1101.100000,0.680000
50%,588.000000,10.000000,31.000000,1564.300000,0.730000
75%,1029.300000,11.000000,46.500000,2925.550000,0.875000
max,1199.200000,12.000000,49.000000,3272.100000,0.980000


In [7]:
# 3.3  Unique-value check for categorical columns
for col in ["source_type", "language", "status"]:
    vals = df[col].value_counts(dropna=False)
    print(f"\n--- {col} ({df[col].nunique()} unique, {df[col].isna().sum()} NaN) ---")
    print(vals.to_string())


--- source_type (4 unique, 0 NaN) ---
source_type
pdf      15
image    15
audio    15
web      15

--- language (8 unique, 10 NaN) ---
language
None    10
de      10
en       7
bs       7
 bs      6
EN       6
  en     5
sr       5
hr       4

--- status (3 unique, 8 NaN) ---
status
extracted    24
processed    20
None          8
reviewed      8


In [8]:
# 3.4  Check for whitespace issues in author names
authors = df["metadata.author"].dropna()
leading_trailing = authors[authors != authors.str.strip()]
print(f"Authors with leading/trailing whitespace: {len(leading_trailing)}")
if len(leading_trailing) > 0:
    for a in leading_trailing.values[:5]:
        print(f"  '{a}'")

Authors with leading/trailing whitespace: 23
  '  Smith, John '
  'Kovačević, Marko '
  ' Hadžić, Lejla'
  'Kovačević, Marko '
  'Kovačević, Marko '


In [9]:
# 3.5  Record the audit baseline — we will compare after cleaning
baseline = {
    "total_missing": int(df.isna().sum().sum()),
    "rows": len(df),
    "cols": len(df.columns),
    "duplicated_rows": int(df.duplicated().sum()),
}
print("Baseline audit:", baseline)

Baseline audit: {'total_missing': 160, 'rows': 60, 'cols': 13, 'duplicated_rows': 0}


---
## 4 — Handling missing data

### 4.1 Missingness patterns (theory)

Not all missing values are the same:

| Pattern | Meaning | Example |
|---------|---------|---------|
| **MCAR** — Missing Completely At Random | Missingness is random, unrelated to any variable | A sensor randomly fails |
| **MAR** — Missing At Random | Missingness depends on *observed* variables | `metadata.pages` is only defined for PDFs |
| **MNAR** — Missing Not At Random | Missingness depends on the *missing value itself* | High-income people skip the income question |

In our data, many missings are **MAR** — structural. `metadata.pages` is `NaN`
for audio and web sources because page count makes no sense there.
`metadata.duration_sec` is `NaN` for PDFs because PDFs have no duration.

A few missings are **data-quality issues**: `file_size_kb` and `language`
should have been recorded for every document but weren't.

### 4.2 Strategy: structural missings — leave or fill with a sentinel

In [10]:
# Structural missings: pages is NaN for non-PDF sources — correct!
print("Pages per source_type:")
print(
    df.groupby("source_type", observed=True)["metadata.pages"]
    .agg(total="size", with_value="count")
)

Pages per source_type:
             total  with_value
source_type                   
audio           15           0
image           15           0
pdf             15          15
web             15           0


In [11]:
# For analysis we might fill structural NaNs with 0 or a sentinel.
# But we should NOT drop these rows — the documents are valid.

# Example: fill pages with 0 for non-PDF sources
df["metadata.pages"] = df["metadata.pages"].fillna(0).astype(int)
print("Pages after fillna(0):")
print(df["metadata.pages"].value_counts().sort_index().head(10))

Pages after fillna(0):
metadata.pages
0     45
5      2
7      2
15     1
16     1
21     1
31     1
36     1
37     1
46     1
Name: count, dtype: int64


### 4.3 `dropna()` — removing rows or columns

In [12]:
# dropna() removes rows (axis=0) or columns (axis=1) with any NaN.
# CAREFUL: this can throw away a lot of data.

# How many rows would we lose if we dropped every row with any NaN?
before = len(df)
after = len(df.dropna())
print(f"dropna() would keep {after}/{before} rows — we'd lose {before - after}")
print("That is too aggressive. Let's be more targeted.")

dropna() would keep 0/60 rows — we'd lose 60
That is too aggressive. Let's be more targeted.


In [13]:
# Drop rows only where SPECIFIC important columns are missing.
# 'language' is a data-quality missing — we might drop those rows,
# or we might fill them. Let's count first.

print(f"Rows missing 'language': {df['language'].isna().sum()}")
print(f"Rows missing 'file_size_kb': {df['file_size_kb'].isna().sum()}")

Rows missing 'language': 10
Rows missing 'file_size_kb': 3


In [14]:
# dropna with subset — only drop if language is missing
df_no_missing_lang = df.dropna(subset=["language"])
print(f"After dropping missing-language rows: {len(df_no_missing_lang)} rows")
# We are NOT assigning back to df yet — just demonstrating.

After dropping missing-language rows: 50 rows


In [15]:
# thresh parameter: keep rows that have at least N non-null values
# Useful as a safety net against very sparse rows.
threshold = len(df.columns) - 3  # tolerate up to 3 missing values
df_thresh = df.dropna(thresh=threshold)
print(f"Rows with at most 3 NaN: {len(df_thresh)}/{len(df)}")

Rows with at most 3 NaN: 58/60


### 4.4 `fillna()` — filling missing values

In [16]:
# Fill missing language with "unknown"
print("Before:", df["language"].isna().sum(), "missing")
df["language"] = df["language"].fillna("unknown")
print("After:", df["language"].isna().sum(), "missing")
print(df["language"].value_counts())

Before: 10 missing
After: 0 missing
language
unknown    10
de         10
en          7
bs          7
 bs         6
EN          6
  en        5
sr          5
hr          4
Name: count, dtype: int64


In [17]:
# Fill missing file_size_kb with the median (robust to outliers)
median_size = df["file_size_kb"].median()
print(f"Median file_size_kb: {median_size}")
df["file_size_kb"] = df["file_size_kb"].fillna(median_size)
print(f"Missing file_size_kb after fill: {df['file_size_kb'].isna().sum()}")

Median file_size_kb: 588.0
Missing file_size_kb after fill: 0


In [18]:
# Forward fill (ffill) and backward fill (bfill) — useful for time series.
# Not ideal for our document data, but let's see what it does.

demo = pd.Series([1, np.nan, np.nan, 4, np.nan, 6])
print("Original:     ", demo.tolist())
print("Forward fill: ", demo.ffill().tolist())
print("Backward fill:", demo.bfill().tolist())

Original:      [1.0, nan, nan, 4.0, nan, 6.0]
Forward fill:  [1.0, 1.0, 1.0, 4.0, 4.0, 6.0]
Backward fill: [1.0, 4.0, 4.0, 4.0, 6.0, 6.0]


### 4.5 `interpolate()` — estimating missing values

In [19]:
# interpolate() fills NaN by interpolating between neighbours.
# Makes sense for numeric, ordered data (e.g. time series).

demo_interp = pd.Series([10, np.nan, np.nan, 40, np.nan, 60])
print("Original:     ", demo_interp.tolist())
print("Interpolated: ", demo_interp.interpolate().tolist())
# 10, 20, 30, 40, 50, 60  — linear interpolation filled the gaps.

Original:      [10.0, nan, nan, 40.0, nan, 60.0]
Interpolated:  [10.0, 20.0, 30.0, 40.0, 50.0, 60.0]


### 4.6 When to drop vs. when to fill

| Situation | Action |
|-----------|--------|
| Very few rows missing (<5 %) and random | **Drop** — little data loss |
| Structural missing (MAR) | **Fill** with sentinel (0, "N/A") or **leave** |
| Important column, many missing | **Fill** with median/mode/domain value |
| Most values missing in a column | **Drop the column** |
| Time series with gaps | **Interpolate** or **ffill** |

The right choice always depends on **domain knowledge**.

In [20]:
# Fill structural NaN for duration_sec and ocr_confidence
df["metadata.duration_sec"] = df["metadata.duration_sec"].fillna(0)
df["metadata.ocr_confidence"] = df["metadata.ocr_confidence"].fillna(0)

# Fill missing status with "unknown"
df["status"] = df["status"].fillna("unknown")

# Check how many NaN remain
remaining = df.isna().sum()
print("Remaining missing values:")
print(remaining[remaining > 0])
if remaining.sum() == 0:
    print("\nNo missing values remain!")

Remaining missing values:
metadata.author    4
dtype: int64


---
## 5 — Data transformation: `map`, `apply`, `replace`, lambda

These are your Swiss Army knives for reshaping values.

### 5.1 `map()` — element-wise mapping with a dict or function

Works on a **Series**. Pass a dict to recode values, or a function to transform each element.

In [21]:
# Recode source_type to more descriptive labels using a dict
label_map = {
    "pdf": "PDF Document",
    "image": "Image / OCR",
    "audio": "Audio Transcript",
    "web": "Web Page",
}

df["source_label"] = df["source_type"].map(label_map)
print(df[["source_type", "source_label"]].drop_duplicates())

  source_type      source_label
0         pdf      PDF Document
1       image       Image / OCR
2       audio  Audio Transcript
3         web          Web Page


In [22]:
# map with a function — classify documents by size
def size_category(kb):
    if kb < 100:
        return "small"
    elif kb < 500:
        return "medium"
    else:
        return "large"

df["size_cat"] = df["file_size_kb"].map(size_category)
print(df["size_cat"].value_counts())

size_cat
large     38
medium    21
small      1
Name: count, dtype: int64


### 5.2 `apply()` — row or column operations

Works on a **DataFrame** (axis=0 for columns, axis=1 for rows).
Slower than vectorised ops — use only when needed.

In [23]:
# apply on a column (Series) — same as map for simple functions
df["title_length"] = df["title"].apply(len)
print(df[["title", "title_length"]].head())

                        title  title_length
0    Annual Report 2024 — IBU            24
1             Campus Map Scan            15
2  Lecture Recording — Week 3            26
3       IBU Homepage Snapshot            21
4      Financial Statement Q3            22


In [24]:
# apply across rows (axis=1) — combine multiple columns
def doc_summary(row):
    return f"{row['source_type'].upper()}: {row['title'][:30]}..."

df["summary"] = df.apply(doc_summary, axis=1)
print(df["summary"].head())

0        PDF: Annual Report 2024 — IBU...
1               IMAGE: Campus Map Scan...
2    AUDIO: Lecture Recording — Week 3...
3           WEB: IBU Homepage Snapshot...
4          PDF: Financial Statement Q3...
Name: summary, dtype: object


### 5.3 `replace()` — substituting values

In [25]:
# Replace specific values
df["language"] = df["language"].replace({
    "unknown": "und",  # ISO 639 code for undetermined
})
print(df["language"].value_counts())

language
und     10
de      10
en       7
bs       7
 bs      6
EN       6
  en     5
sr       5
hr       4
Name: count, dtype: int64


### 5.4 Lambda functions — quick inline transforms

In [26]:
# Lambda — one-line throwaway function
df["title_upper"] = df["title"].apply(lambda t: t.upper())
print(df["title_upper"].head(3))

# Clean up demo columns we don't need going forward
df.drop(columns=["source_label", "size_cat", "title_length",
                  "summary", "title_upper"], inplace=True)

0      ANNUAL REPORT 2024 — IBU
1               CAMPUS MAP SCAN
2    LECTURE RECORDING — WEEK 3
Name: title_upper, dtype: object


---
## 6 — String cleaning with the `.str` accessor

Text columns are where most of the mess lives. pandas gives every string
column a `.str` accessor with dozens of methods — all vectorised, so no
loops needed.

### 6.1 Whitespace: `str.strip()`, `str.lstrip()`, `str.rstrip()`

In [27]:
# Clean author names — remove leading/trailing whitespace
print("Before strip (first 5 non-null):")
print(df["metadata.author"].dropna().head().tolist())

df["metadata.author"] = df["metadata.author"].str.strip()

print("\nAfter strip:")
print(df["metadata.author"].dropna().head().tolist())

Before strip (first 5 non-null):
['Hodžić, Amina', '  Smith, John ', 'Kovačević, Marko ', 'Doe, Jane', 'Bešić,  Adnan']

After strip:
['Hodžić, Amina', 'Smith, John', 'Kovačević, Marko', 'Doe, Jane', 'Bešić,  Adnan']


### 6.2 Case normalisation: `str.lower()`, `str.upper()`, `str.title()`

In [28]:
# Normalise language codes to lowercase
print("Before:", df["language"].unique())
df["language"] = df["language"].str.lower()
print("After: ", df["language"].unique())

Before: ['en' ' bs ' 'und' 'bs' 'EN' '  en' 'sr' 'de' 'hr']
After:  ['en' ' bs ' 'und' 'bs' '  en' 'sr' 'de' 'hr']


### 6.3 `str.replace()` — simple and regex

In [29]:
# Collapse multiple internal spaces in author names
df["metadata.author"] = (
    df["metadata.author"]
    .str.replace(r"\s+", " ", regex=True)
)

# Verify
print(df["metadata.author"].dropna().head(5).tolist())

['Hodžić, Amina', 'Smith, John', 'Kovačević, Marko', 'Doe, Jane', 'Bešić, Adnan']


### 6.4 Other useful `.str` methods

In [30]:
# str.contains — find documents mentioning "Bosn" in the text
bosnia_docs = df[df["content.text"].str.contains("Bosn", case=False, na=False)]
print(f"Documents mentioning Bosnia: {len(bosnia_docs)}")
print(bosnia_docs[["title", "source_type"]].head())

Documents mentioning Bosnia: 0
Empty DataFrame
Columns: [title, source_type]
Index: []


In [31]:
# str.len — title character lengths
print(df["title"].str.len().describe())

count    60.000000
mean     24.733333
std       4.095747
min      15.000000
25%      22.000000
50%      24.000000
75%      27.250000
max      35.000000
Name: title, dtype: float64


In [32]:
# str.startswith / str.endswith
pdf_titles = df[df["title"].str.endswith(".pdf", na=False)]
print(f"Titles ending in '.pdf': {len(pdf_titles)}")

Titles ending in '.pdf': 0


---
## 7 — Regular expressions for advanced cleaning

When simple string methods aren't enough, regex gives you precision.
pandas `.str` methods accept `regex=True` (or use `str.extract` / `str.extractall`).

In [33]:
# 7.1  Extract prices in KM (Bosnian currency) from text
price_pattern = r"(\d+[.,]?\d*)\s?KM"

df["extracted_price_km"] = df["content.text"].str.extract(
    price_pattern, expand=False
)

prices = df.loc[df["extracted_price_km"].notna(),
                ["title", "content.text", "extracted_price_km"]]
print(f"Documents with KM prices: {len(prices)}")
print(prices.head())

Documents with KM prices: 6
                                  title                                                 content.text  \
3                 IBU Homepage Snapshot  U izvještaju se navodi da je budžet za 2024. godinu 12.5...   
6                 Interview with Alumni         Cijena karte za konferenciju iznosi 150 KM po osobi.   
23  Blog Post — Study Tips for Students  U izvještaju se navodi da je budžet za 2024. godinu 12.5...   
26           Conference Talk — NLP 2024         Cijena karte za konferenciju iznosi 150 KM po osobi.   
43          Weather Forecast — Sarajevo  U izvještaju se navodi da je budžet za 2024. godinu 12.5...   

   extracted_price_km  
3              12.500  
6                 150  
23             12.500  
26                150  
43             12.500  


In [34]:
# 7.2  Standardise date-like strings in text
# Find dates in DD.MM.YYYY or DD/MM/YYYY format
date_pattern = r"\d{1,2}[./]\d{1,2}[./]\d{2,4}"
has_date = df["content.text"].str.contains(date_pattern, na=False)
print(f"Documents with date-like patterns in text: {has_date.sum()}")

Documents with date-like patterns in text: 3


In [35]:
# 7.3  Remove HTML tags if any leaked into text (common with web sources)
html_tag_pattern = r"<[^>]+>"
before_tags = df["content.text"].str.contains(html_tag_pattern, na=False).sum()
df["content.text"] = df["content.text"].str.replace(html_tag_pattern, "", regex=True)
after_tags = df["content.text"].str.contains(html_tag_pattern, na=False).sum()
print(f"Rows with HTML tags: {before_tags} -> {after_tags}")

# Clean up the temp column
df.drop(columns=["extracted_price_km"], inplace=True)

Rows with HTML tags: 3 -> 0


---
## 8 — Removing duplicates

### 8.1 Exact duplicates

In [36]:
# Check for fully duplicated rows
dup_count = df.duplicated().sum()
print(f"Fully duplicated rows: {dup_count}")

Fully duplicated rows: 0


In [37]:
# Check for duplicates on specific columns — maybe same title + source?
dup_title_source = df.duplicated(subset=["title", "source_type"], keep=False)
print(f"Rows with duplicate title + source_type: {dup_title_source.sum()}")

if dup_title_source.sum() > 0:
    print(df.loc[dup_title_source, ["title", "source_type"]].sort_values("title"))

Rows with duplicate title + source_type: 2
                       title source_type
0   Annual Report 2024 — IBU         pdf
56  Annual Report 2024 — IBU         pdf


In [38]:
# drop_duplicates — keep first occurrence
before = len(df)
df = df.drop_duplicates(subset=["title", "source_type"], keep="first")
after = len(df)
print(f"Removed {before - after} duplicate(s). Rows: {before} -> {after}")

Removed 1 duplicate(s). Rows: 60 -> 59


### 8.2 Fuzzy deduplication (concept)

Sometimes duplicates aren't exact — "Report 2024" vs "report_2024" vs
"Report  2024". Libraries like **rapidfuzz** (or the older fuzzywuzzy)
compute string similarity scores:

```python
from rapidfuzz import fuzz
fuzz.ratio("Report 2024", "report_2024")  # → ~85
```

For this course, knowing the concept is enough. In practice, fuzzy
deduplication is a common step in real pipelines.

---
## 9 — Data type conversion

Correct dtypes are essential — you cannot compute statistics on strings,
and date arithmetic requires datetime objects.

In [39]:
# Current dtypes
print(df.dtypes)

_id                         object
title                       object
source_type                 object
extracted_at                object
file_size_kb               float64
language                    object
status                      object
content.text                object
content.word_count           int64
metadata.author             object
metadata.pages               int64
metadata.duration_sec      float64
metadata.ocr_confidence    float64
dtype: object


In [40]:
# 9.1  astype() — explicit conversion
# Ensure word_count is integer (it should be, but let's be safe)
df["content.word_count"] = pd.to_numeric(df["content.word_count"], errors="coerce")
df["content.word_count"] = df["content.word_count"].fillna(0).astype(int)
print(f"word_count dtype: {df['content.word_count'].dtype}")

word_count dtype: int64


In [41]:
# 9.2  to_datetime() — parse date strings
df["extracted_at"] = pd.to_datetime(df["extracted_at"], errors="coerce")
print(f"extracted_at dtype: {df['extracted_at'].dtype}")
print(df["extracted_at"].head(3))

extracted_at dtype: datetime64[ns]
0   2025-02-11 03:00:00
1   2024-09-27 21:00:00
2   2024-10-26 07:00:00
Name: extracted_at, dtype: datetime64[ns]


In [42]:
# 9.3  Date components — now that we have datetime, we can extract parts
df["extracted_year"] = df["extracted_at"].dt.year
df["extracted_month"] = df["extracted_at"].dt.month
df["extracted_dow"] = df["extracted_at"].dt.day_name()
print(df[["extracted_at", "extracted_year", "extracted_month", "extracted_dow"]].head())

         extracted_at  extracted_year  extracted_month extracted_dow
0 2025-02-11 03:00:00            2025                2       Tuesday
1 2024-09-27 21:00:00            2024                9        Friday
2 2024-10-26 07:00:00            2024               10      Saturday
3 2025-02-27 17:00:00            2025                2      Thursday
4 2024-09-02 05:00:00            2024                9        Monday


In [43]:
# 9.4  to_numeric() with errors='coerce' — safely convert messy numbers
messy_numbers = pd.Series(["42", "3.14", "N/A", "100", ""])
clean_numbers = pd.to_numeric(messy_numbers, errors="coerce")
print("Messy:  ", messy_numbers.tolist())
print("Clean:  ", clean_numbers.tolist())
print("NaN count:", clean_numbers.isna().sum())

Messy:   ['42', '3.14', 'N/A', '100', '']
Clean:   [42.0, 3.14, nan, 100.0, nan]
NaN count: 2


In [44]:
# 9.5  pd.Categorical — for repeated string columns (memory + speed)
for col in ["source_type", "language", "status"]:
    df[col] = df[col].astype("category")

print("Category columns:")
print(df[["source_type", "language", "status"]].dtypes)

Category columns:
source_type    category
language       category
status         category
dtype: object


---
## 10 — Validating cleaned data

After cleaning, **prove** the data meets expectations. Use assert statements
for automated checks.

In [45]:
# 10.1  No missing values in critical columns
critical_cols = ["title", "source_type", "language", "content.text"]
for col in critical_cols:
    n_missing = df[col].isna().sum()
    assert n_missing == 0, f"{col} still has {n_missing} missing values!"
    print(f"  ✓ {col}: no missing values")

  ✓ title: no missing values
  ✓ source_type: no missing values
  ✓ language: no missing values
  ✓ content.text: no missing values


In [46]:
# 10.2  Numeric ranges make sense
assert (df["content.word_count"] >= 0).all(), "Negative word counts found!"
print("  ✓ word_count: all >= 0")

assert (df["file_size_kb"] > 0).all(), "Non-positive file sizes found!"
print("  ✓ file_size_kb: all > 0")

assert (df["metadata.pages"] >= 0).all(), "Negative page counts found!"
print("  ✓ metadata.pages: all >= 0")

  ✓ word_count: all >= 0
  ✓ file_size_kb: all > 0
  ✓ metadata.pages: all >= 0


In [47]:
# 10.3  No duplicate titles within same source type
dup_check = df.duplicated(subset=["title", "source_type"])
assert dup_check.sum() == 0, f"Found {dup_check.sum()} duplicates!"
print(f"  ✓ No duplicate title+source_type pairs")

  ✓ No duplicate title+source_type pairs


In [48]:
# 10.4  Dates are reasonable (not in the future, not before 2020)
now = pd.Timestamp.now()
valid_dates = df["extracted_at"].dropna()
assert (valid_dates <= now).all(), "Future dates found!"
assert (valid_dates >= pd.Timestamp("2020-01-01")).all(), "Dates before 2020!"
print(f"  ✓ All dates between 2020 and now")

  ✓ All dates between 2020 and now


In [49]:
# 10.5  Compare with baseline
final_missing = int(df.isna().sum().sum())
print(f"\n=== Cleaning summary ===")
print(f"Missing values: {baseline['total_missing']} -> {final_missing}")
print(f"Rows: {baseline['rows']} -> {len(df)}")
print(f"Columns: {baseline['cols']} -> {len(df.columns)}")


=== Cleaning summary ===
Missing values: 160 -> 4
Rows: 60 -> 59
Columns: 13 -> 16


---
## 11 — Testing with pytest

### 11.1 Why test cleaning code?

Cleaning code is **logic** — it makes decisions (drop this, fill that, transform the other).
Logic has bugs. Tests catch them.

**pytest** is the standard Python testing framework:
- Test functions start with `test_`
- Use plain `assert` statements
- Run with `pytest` from the terminal

### 11.2 Writing tests

You write tests in a separate `.py` file. Here is what `test_cleaning.py` might look like:

In [50]:
# We'll write a test file to disk and run it with pytest.
# First, let's define a simple cleaning function.

def clean_author(name):
    """Strip whitespace and collapse internal spaces."""
    if pd.isna(name):
        return name
    return re.sub(r"\s+", " ", name.strip())

# Quick manual check
print(clean_author("  Dženana   Mehmedović  "))  # -> "Dženana Mehmedović"
print(clean_author(None))                         # -> None

Dženana Mehmedović
None


In [51]:
# Write a test file
test_lines = [
    'import pandas as pd',
    'import numpy as np',
    'import re',
    '',
    'def clean_author(name):',
    '    # Strip whitespace and collapse internal spaces.',
    '    if pd.isna(name):',
    '        return name',
    '    return re.sub(r"\\s+", " ", name.strip())',
    '',
    'def clean_language(lang):',
    '    # Lowercase and fill missing.',
    '    if pd.isna(lang):',
    '        return "und"',
    '    return lang.strip().lower()',
    '',
    '',
    'class TestCleanAuthor:',
    '    def test_strips_whitespace(self):',
    '        assert clean_author("  Alice  ") == "Alice"',
    '',
    '    def test_collapses_internal_spaces(self):',
    '        assert clean_author("Bob   Smith") == "Bob Smith"',
    '',
    '    def test_handles_none(self):',
    '        assert pd.isna(clean_author(None))',
    '',
    '    def test_handles_nan(self):',
    '        assert pd.isna(clean_author(float("nan")))',
    '',
    '    def test_normal_name_unchanged(self):',
    '        assert clean_author("Élise Müller") == "Élise Müller"',
    '',
    '',
    'class TestCleanLanguage:',
    '    def test_lowercase(self):',
    '        assert clean_language("EN") == "en"',
    '',
    '    def test_fills_none(self):',
    '        assert clean_language(None) == "und"',
    '',
    '    def test_strips_and_lowers(self):',
    '        assert clean_language("  BS ") == "bs"',
]

with open("test_cleaning.py", "w", encoding="utf-8") as f:
    f.write("\n".join(test_lines) + "\n")

print("test_cleaning.py written.")

test_cleaning.py written.


In [52]:
!pip install pytest


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [53]:
# Run pytest using the same interpreter the notebook is running on.
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_cleaning.py", "-v"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-9.0.1, pluggy-1.6.0 -- /Library/Frameworks/Python.framework/Versions/3.13/bin/python3
cachedir: .pytest_cache
rootdir: /Users/dzelila/Downloads/unstructured-data
plugins: anyio-4.11.0
collecting ... collected 8 items

test_cleaning.py::TestCleanAuthor::test_strips_whitespace PASSED         [ 12%]
test_cleaning.py::TestCleanAuthor::test_collapses_internal_spaces PASSED [ 25%]
test_cleaning.py::TestCleanAuthor::test_handles_none PASSED              [ 37%]
test_cleaning.py::TestCleanAuthor::test_handles_nan PASSED               [ 50%]
test_cleaning.py::TestCleanAuthor::test_normal_name_unchanged PASSED     [ 62%]
test_cleaning.py::TestCleanLanguage::test_lowercase PASSED               [ 75%]
test_cleaning.py::TestCleanLanguage::test_fills_none PASSED              [ 87%]
test_cleaning.py::TestCleanLanguage::test_strips_and_lowers PASSED       [100%]

===================

### 11.3 Key pytest concepts

| Concept | Example |
|---------|---------|
| Test function | `def test_something():` |
| Assert | `assert result == expected` |
| Test class | `class TestSomething:` groups related tests |
| Fixture | `@pytest.fixture` — reusable setup data |
| Run | `pytest test_file.py -v` |

For this course, simple `assert`-based tests are enough. In larger
projects you would use fixtures, parametrize, and mocking.

---
## 12 — Reproducible cleaning pipeline

All the cleaning we did above was cell-by-cell. In a real project,
wrap it in a **function** so you can re-run it on new data with one call.

In [54]:
def clean_documents(df_input):
    """
    Full cleaning pipeline for extracted-documents data.

    Takes a raw DataFrame (from json_normalize) and returns a clean copy.
    Every decision is documented in comments.
    """
    df = df_input.copy()

    # --- 1. Structural missing values ---
    # pages, duration, ocr_confidence: fill with 0 (not applicable for some source types)
    df["metadata.pages"] = df["metadata.pages"].fillna(0).astype(int)
    df["metadata.duration_sec"] = df["metadata.duration_sec"].fillna(0)
    df["metadata.ocr_confidence"] = df["metadata.ocr_confidence"].fillna(0)

    # --- 2. Data-quality missing values ---
    df["language"] = df["language"].fillna("unknown").str.strip().str.lower()
    df["language"] = df["language"].replace({"unknown": "und"})
    df["file_size_kb"] = df["file_size_kb"].fillna(df["file_size_kb"].median())
    df["status"] = df["status"].fillna("unknown")

    # --- 3. String cleaning ---
    df["metadata.author"] = (
        df["metadata.author"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    df["content.text"] = df["content.text"].str.replace(r"<[^>]+>", "", regex=True)

    # --- 4. Type conversion ---
    df["content.word_count"] = pd.to_numeric(df["content.word_count"], errors="coerce").fillna(0).astype(int)
    df["extracted_at"] = pd.to_datetime(df["extracted_at"], errors="coerce")

    # --- 5. Duplicates ---
    df = df.drop_duplicates(subset=["title", "source_type"], keep="first")

    # --- 6. Category dtypes for memory ---
    for col in ["source_type", "language", "status"]:
        df[col] = df[col].astype("category")

    # --- 7. Reset index ---
    df = df.reset_index(drop=True)

    return df

In [55]:
# Test the pipeline on our raw data
df_clean = clean_documents(df_raw)

print(f"Shape: {df_clean.shape}")
print(f"Missing values: {df_clean.isna().sum().sum()}")
print(f"Dtypes:\n{df_clean.dtypes}")

Shape: (59, 13)
Missing values: 4
Dtypes:
_id                                object
title                              object
source_type                      category
extracted_at               datetime64[ns]
file_size_kb                      float64
language                         category
status                           category
content.text                       object
content.word_count                  int64
metadata.author                    object
metadata.pages                      int64
metadata.duration_sec             float64
metadata.ocr_confidence           float64
dtype: object


In [56]:
# Save cleaned data for Week 10
df_clean.to_csv("documents_cleaned.csv", index=False)
df_clean.to_json("documents_cleaned.json", orient="records", indent=2,
                 date_format="iso")
print("Saved: documents_cleaned.csv and documents_cleaned.json")

Saved: documents_cleaned.csv and documents_cleaned.json


---
## 13 — Summary

What we did in this notebook:

1. **Audit** — measured missing values, checked dtypes, spotted whitespace and duplicates
2. **Missing data** — distinguished structural (MAR) from quality issues; used `fillna()`, `dropna()`, `interpolate()`
3. **Transformation** — `map()` for recoding, `apply()` for row-level logic, `replace()` for substitution, lambda for one-liners
4. **String cleaning** — `.str.strip()`, `.str.lower()`, `.str.replace()` with regex
5. **Regex** — extracted prices, found dates, stripped HTML tags
6. **Duplicates** — `duplicated()` + `drop_duplicates()` with `subset` and `keep`
7. **Type conversion** — `to_numeric()`, `to_datetime()`, `astype()`, `pd.Categorical`
8. **Validation** — assert-based checks on ranges, completeness, uniqueness, dates
9. **Testing** — wrote and ran pytest tests for cleaning functions
10. **Pipeline** — wrapped everything in a reusable `clean_documents()` function

---

### What is next — Week 10: Data Analysis and Aggregation

The data is clean. Now we **analyse** it: combine datasets with `merge` and `concat`,
aggregate with `groupby`, pivot and reshape, work with dates and time series,
and connect to MySQL alongside MongoDB.

---
*IT 2012 – Unstructured Data · Week 9 · International Burch University*